In [1]:
# ============================================
# 0. 載入套件與環境設定
# ============================================

import os
import sys

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from train_save import train_and_save_model

print("環境準備完畢，已成功載入 train_and_save_model 模組。")


環境準備完畢，已成功載入 train_and_save_model 模組。


## Part 1：定義請求與回應的 Pydantic 模型

In [2]:
# ============================================
# 定義 Pydantic 訓練相關模型（與 app.py 相同）
# ============================================

from pydantic import BaseModel,Field
from pprint import pprint

class TrainConfig(BaseModel):
    test_size: float = Field(0.2, description="測試集分割比例", ge=0.1 , le=0.5)
    random_state: int = Field(76, description="隨機種子", ge=0)
    model_type: str = Field("LinearRegression", description="模型演算法類型 (LinearRegression, Lasso, Ridge)")
    alpha: float = Field(1.0, description="正則化強度 alpha (適用於 Lasso 與 Ridge)", ge= 0.001, le=100.0)

class TrainResult(BaseModel):
    status: str = Field(..., description="執行結果狀態")
    r2: float = Field(..., description="測試集 R-squared 決定係數")
    coef: list[float] = Field(..., description="特徵權重係數列表")
    intercept: float = Field(..., description="截距")
    feature_coefs: dict[str, float] = Field(..., description="特徵及其權重映射")
    model_type: str = Field(..., description="模型演算法類型")
    alpha: float = Field(..., description="正則化強度 alpha")
    train_time: float = Field(..., description="訓練耗時 (秒)")
    message:str = Field(..., description="提示訊息")

print("TranConfig(BaseModel)")
pprint(TrainConfig.model_json_schema())
print("==============================")
print("TrainResult(BaseModel)")
pprint(TrainResult.model_json_schema())

TranConfig(BaseModel)
{'properties': {'alpha': {'default': 1.0,
                          'description': '正則化強度 alpha (適用於 Lasso 與 Ridge)',
                          'maximum': 100.0,
                          'minimum': 0.001,
                          'title': 'Alpha',
                          'type': 'number'},
                'model_type': {'default': 'LinearRegression',
                               'description': '模型演算法類型 (LinearRegression, '
                                              'Lasso, Ridge)',
                               'title': 'Model Type',
                               'type': 'string'},
                'random_state': {'default': 76,
                                 'description': '隨機種子',
                                 'minimum': 0,
                                 'title': 'Random State',
                                 'type': 'integer'},
                'test_size': {'default': 0.2,
                              'description': '測試集分割比例',
              

## 拆解 train_and_save_model() 底層訓練與序列化

In [3]:
from train_save import train_and_save_model
res_ridge:dict = train_and_save_model(
    test_size=0.2,
    random_state=76,
    model_type="Ridge",
    alpha=10.0
)
pprint(res_ridge)

開始訓練 Ridge 嶺迴歸(α=10.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 C:\Users\User\Documents\GitHub\2026_07_03_tvdi_AI\backend\08_07\salary_model.joblib...
模型儲存成功！
{'alpha': 10.0,
 'coef': [3.915606705818322,
          10.029103401270465,
          -1.4644383465780102,
          -1.182860911975303,
          2.1482340576072554],
 'feature_coefs': {'City_城市A': -1.4644383465780102,
                   'City_城市B': -1.182860911975303,
                   'City_城市C': 2.1482340576072554,
                   'EducationLevel': 10.029103401270465,
                   'YearsExperience': 3.915606705818322},
 'intercept': 51.228571428571435,
 'message': 'Ridge 嶺迴歸(α=10.0) 模型訓練完成並儲存成功！',
 'model_type': 'Ridge',
 'r2': 0.8253872705107945,
 'status': 'success',
 'train_time': 0.05635213851928711}


## 理解 load_model_state() 全域動態更新機制

In [4]:
import joblib

current_dir = os.getcwd()
model_path = os.path.join(current_dir, "salary_model.joblib")
MODEL_STATE = {}

def load_model_state():
    global MODEL_STATE
    if not os.path.exists(model_path):
        train_and_save_model()

    model_data = joblib.load(model_path)
    MODEL_STATE.clear()
    MODEL_STATE.update(
        {
            "model": model_data["model"],
            "oe": model_data["oe"],
            "ohe": model_data["ohe"],
            "scaler": model_data["scaler"],
            "r2": model_data.get("r2"),
            "feature_names": model_data["feature_names"],
            "feature_coefs": model_data.get("feature_coefs",{}),
            "model_type": model_data.get("model_type"),
            "alpha": model_data.get("alpha")
        }
    )
    print(f"✅ MODEL_STATE 已成功更新！當前模型：{MODEL_STATE['model_type']}，R² Score：{MODEL_STATE['r2']:.4f}")

load_model_state()

✅ MODEL_STATE 已成功更新！當前模型：Ridge，R² Score：0.8254


## 流程串成 train_api 函數

In [5]:
from fastapi import HTTPException

def train_api(config:TrainConfig) -> dict:
    """
    訓練端點：傳入測試集比例、隨機種子、模型類型與 alpha，線上重新訓練模型，並即時更新服務所使用的模型。
    """
    try:
        # 1. 執行重新訓練並儲存模型
        res = train_and_save_model(
            test_size=config.test_size,
            random_state= config.random_state,
            model_type= config.model_type,
            alpha=config.alpha
        )
         # 2. 線上重新載入最新模型狀態至全域變數
        load_model_state()
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"線上訓練失敗: {str(e)}")

    return res
    

##  FastAPI TestClient 整合測試

In [6]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

mini_api = FastAPI()
@mini_api.post("/train", response_model=TrainResult)
def train_endpoint(config:TrainConfig):
    res = train_api(config=config)
    return res

client = TestClient(mini_api)
response = client.post("/train", json={
    "test_size": 0.2,
    "random_state": 76,
    "model_type": "Lasso",
    "alpha": 5.0
})
print("【重訓 Lasso 結果】")
print("HTTP 狀態碼:", response.status_code)
pprint(response.json())


開始訓練 Lasso 迴歸(α=5.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 C:\Users\User\Documents\GitHub\2026_07_03_tvdi_AI\backend\08_07\salary_model.joblib...
模型儲存成功！
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
【重訓 Lasso 結果】
HTTP 狀態碼: 200
{'alpha': 5.0,
 'coef': [0.21476373685969363, 11.506443086096304, -0.0, -0.0, 0.0],
 'feature_coefs': {'City_城市A': -0.0,
                   'City_城市B': -0.0,
                   'City_城市C': 0.0,
                   'EducationLevel': 11.506443086096304,
                   'YearsExperience': 0.21476373685969363},
 'intercept': 51.228571428571435,
 'message': 'Lasso 迴歸(α=5.0) 模型訓練完成並儲存成功！',
 'model_type': 'Lasso',
 'r2': 0.8441175875255694,
 'status': 'success',
 'train_time': 0.012518644332885742}


C:\Users\User\Documents\GitHub\2026_07_03_tvdi_AI\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
